In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [2]:
housing = pd.read_csv("housing.csv")

In [3]:
housing

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


In [4]:
housing["income_category"] = pd.cut(housing["median_income"],bins=[0.0,1.5,3.0,4.5,6.0,np.inf],labels=[1,2,3,4,5])

In [5]:
split = StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=42)

In [6]:
for train_index,test_index in split.split(housing,housing["income_category"]):
    train_set = housing.loc[train_index].drop("income_category",axis=1)
    test_set = housing.loc[test_index].drop("income_category",axis=1)

In [7]:
housing = train_set.copy()

In [8]:
housing_labels = housing["median_house_value"].copy()
housing = housing.drop("median_house_value",axis=1)

In [9]:
numerical_attribute = housing.drop("ocean_proximity",axis=1).columns.tolist()
categorical_attribute = ["ocean_proximity"]

In [10]:
numerical_pipeline = Pipeline([
    ("imputer" , SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),])

In [11]:
categorical_pipeline = Pipeline([
    ("onehotencoder" , OneHotEncoder(handle_unknown="ignore"))])

In [12]:
full_pipeline = ColumnTransformer([
    ("num" , numerical_pipeline,numerical_attribute),
    ("cat" , categorical_pipeline,categorical_attribute)])

In [13]:
prepared_data = full_pipeline.fit_transform(housing)

In [14]:
print(prepared_data.shape)

(16512, 13)


In [39]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score

In [40]:
# training the model

In [44]:
linear_reg = LinearRegression()
linear_reg.fit(prepared_data,housing_labels)
linear_prediction = linear_reg.predict(prepared_data)
linear_rmse = root_mean_squared_error( housing_labels,linear_prediction)
# linear_rmses = -cross_val_score(linear_reg,prepared_data,housing_labels,scoring="neg_root_mean_squared_error",cv=10)
# print(pd.Series(linear_rmses).describe())

In [45]:
linear_rmse

69050.56219504567

In [46]:
decision_tree = DecisionTreeRegressor()
decision_tree.fit(prepared_data,housing_labels)
decision_prediction = decision_tree.predict(prepared_data)
decision_rmse = root_mean_squared_error(housing_labels,decision_prediction)
# decision_rmses = -cross_val_score(decision_tree,prepared_data,housing_labels,scoring="neg_root_mean_squared_error",cv=10)
# print(pd.Series(decision_rmses).describe())

In [47]:
decision_rmse

0.0

In [48]:
forest_reg = RandomForestRegressor(random_state=42)
forest_reg.fit(prepared_data, housing_labels)
forest_preds = forest_reg.predict(prepared_data)
forest_rmse = root_mean_squared_error(housing_labels, forest_preds)
# forest_rmses = -cross_val_score(forest_reg,prepared_data,housing_labels,scoring="neg_root_mean_squared_error",cv=10)
# print(pd.Series(forest_rmses).describe())

In [49]:
forest_rmse

18342.366362322846